## Classification using Bidirectional Deep RNN: Human or Worm

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchtext.data.utils import get_tokenizer
import numpy as np
from RNNUtils import train_classifier_model, plot_errors_accuracy
import matplotlib.pyplot as plt 
import matplotlib.dates as mdates
from tqdm.notebook import trange

### Load and Prepare the Data
The dataset used here is *Human or Worm* from [Genomic Benchmarks](https://github.com/ML-Bioinfo-CEITEC/genomic_benchmarks). This can be accessed using the Python package `genomic-benchmarks`.

In [ ]:
from genomic_benchmarks.data_check import list_datasets, info
from genomic_benchmarks.dataset_getters.utils import LetterTokenizer, build_vocab, coll_factory
from genomic_benchmarks.dataset_getters.pytorch_datasets import DemoHumanOrWorm

In [ ]:
list_datasets()

In [ ]:
info('demo_human_or_worm')

In [ ]:
train_dset = DemoHumanOrWorm(split='train', version=0)
test_dset = DemoHumanOrWorm(split='test', version=0)

In [ ]:
test_dset

The sequences need to be tokenized

In [ ]:
tokenizer = get_tokenizer(LetterTokenizer())
vocabulary = build_vocab(train_dset, tokenizer, use_padding=False)

print("vocab len:" ,vocabulary.__len__())
vocab_dict = vocabulary.get_stoi()

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
def char_to_int_sequence(data, char_to_int):
    int_data = []
    label_data = []
    for string, label in data:
        int_data.append([char_to_int[char] for char in string])
        label_data.append(label)
    return int_data, label_data

In [ ]:
train_int_data, train_lbl = char_to_int_sequence(train_dset, vocab_dict)
test_int_data, test_lbl = char_to_int_sequence(test_dset, vocab_dict)

In [ ]:
train_int_data_a = np.array(train_int_data)
test_int_data_a = np.array(test_int_data)

In [ ]:
train_int_data_t = torch.tensor(train_int_data_a).to(device)
test_int_data_t = torch.tensor(test_int_data_a).to(device)

In [ ]:
train_one_hot_data = F.one_hot(train_int_data_t).to(torch.float32)
test_one_hot_data = F.one_hot(test_int_data_t).to(torch.float32)

In [ ]:
train_lbl_t = torch.tensor(train_lbl, dtype=torch.float32).to(device)
test_lbl_t = torch.tensor(test_lbl, dtype=torch.float32).to(device)

In [ ]:
batch_size = 512

In [ ]:
def prepare_dataloaders(data, labels, batch_size, drop_last=True):
    dataset = TensorDataset(data, labels)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=drop_last)
    return dataloader

In [ ]:
train_dataloader = prepare_dataloaders(train_one_hot_data, train_lbl_t, batch_size)
test_dataloader = prepare_dataloaders(test_one_hot_data, test_lbl_t, batch_size)

### Define the RNN Model

In [ ]:
class DeepRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=3, dropout=0.5):
        super(DeepRNN, self).__init__()
        # Specify the number of layers and dropout in the RNN
        self.rnn = nn.RNN(input_size, hidden_size, num_layers=num_layers, 
                          batch_first=True, bidirectional=True, dropout=dropout)
        self.linear = nn.Linear(hidden_size * 2, output_size)

    def forward(self, x):
        out, _ = self.rnn(x)
        out = out[:, -1, :]
        out = self.linear(out)
        return out

In [ ]:
input_size = 8
hidden_size = 64
output_size = 1
lr = 0.00001
epochs = 100

In [ ]:
model = DeepRNN(input_size=input_size, hidden_size=hidden_size, output_size=output_size).to(device)
model

### Train Model

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=lr)
loss_fn = torch.nn.BCEWithLogitsLoss()

In [ ]:
train_errors, test_errors, train_accuracies, test_accuracies = train_classifier_model(model, train_dataloader, test_dataloader, 
                                        loss_fn, optimizer, epochs)

### Evaluate Model

In [ ]:
filename = 'genomics_deep_rnn_loss.png'
plot_errors_accuracy(train_errors, test_errors, train_accuracies, test_accuracies, filename)

In [ ]:
y_test = np.empty([])
y_pred = np.empty([])
model.eval()
with torch.no_grad():
    for batch_X, batch_y in test_dataloader:
        outputs = model(batch_X)
        predicted = torch.sigmoid(outputs).squeeze() > 0.5
        y_test = np.append(y_test, batch_y.detach().cpu().numpy())
        y_pred = np.append(y_pred, predicted.detach().cpu().numpy().astype(int))

In [ ]:
y_pred

In [ ]:
# Accuracy
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy = {}".format(accuracy))

In [ ]:
# Balanced Accuracy
from sklearn.metrics import balanced_accuracy_score
balanced_accuracy = balanced_accuracy_score(y_test, y_pred)
print("Balanced Accuracy = {}".format(balanced_accuracy))

In [ ]:
# Precision
from sklearn.metrics import precision_score
precision = precision_score(y_test, y_pred)
print("Precision = {}".format(precision))

In [ ]:
# Recall
from sklearn.metrics import recall_score
recall = recall_score(y_test, y_pred)
print("Recall = {}".format(recall))

In [ ]:
# F1-Score
from sklearn.metrics import f1_score
f1 = f1_score(y_test, y_pred)
print("F1-Score = {}".format(f1))

In [ ]:
# AUC
from sklearn.metrics import roc_auc_score
# Note use of raw probabilities
auc = roc_auc_score(y_test, y_pred)
print("AUC = {}".format(auc))